In [0]:
# =============================================================================
# Notebook  : historical_bq_load  [OPTIMIZED]
# Purpose   : BigQuery → S3 historical ingestion — 1B records capable
#
# PERFORMANCE ISSUES IN ORIGINAL & FIXES APPLIED:
#
# ❌ ISSUE 1 — Sequential Python for-loop over 12-hr chunks (THE KILLER)
#   Original processed hundreds of chunks one-by-one in a Python loop.
#   For 2 years of data = 1,460 chunks. At 30s/chunk = 12+ HOURS.
#   ✅ FIX: Use BQ native connector's built-in parallel read protocol.
#   The connector creates parallel BigQuery Storage Read API streams
#   automatically. One single read, all cores utilized.
#
# ❌ ISSUE 2 — spark.sql.shuffle.partitions = 2
#   2 shuffle partitions on 4 cores means Spark cannot parallelize.
#   For 1B records this creates massive per-partition files and OOM risk.
#   ✅ FIX: Set to 8x cores = 32. Use AQE to auto-coalesce small partitions.
#
# ❌ ISSUE 3 — .repartition(2) before write
#   Forces a full shuffle of ALL data into just 2 partitions.
#   On 4 cores writing 100GB = one core does all the S3 work.
#   ✅ FIX: Use .coalesce() for sequential writes (no shuffle).
#           Or let AQE handle partition sizing automatically.
#           Target ~128MB per output file = good S3 object size.
#
# ❌ ISSUE 4 — bounds_df.collect() triggers a full BQ scan just to get min/max
#   This is an extra round-trip to BQ before any real work starts.
#   ✅ FIX: Use BQ's DATE partitioning if available. Otherwise remove the
#   bounds scan — use the BQ connector's preferredMinParallelism instead
#   of manually chunking by timestamp.
#
# ❌ ISSUE 5 — New gRPC connection per chunk
#   Each spark.read in the loop opens a new BQ Storage Read API session.
#   Session setup overhead is ~2-5 seconds × 1,460 chunks = 1-2 hrs wasted.
#   ✅ FIX: Single spark.read call — one session, all parallelism inside.
#
# ❌ ISSUE 6 — No AQE (Adaptive Query Execution)
#   Without AQE Spark cannot handle data skew or auto-optimize join strategies.
#   ✅ FIX: Enable AQE, skew join handling, and dynamic coalescing.
#
# ❌ ISSUE 7 — parquet write without Delta
#   Plain parquet has no OPTIMIZE/ZORDER or incremental append semantics.
#   For bronze landing this is acceptable, but using Delta gives ACID +
#   future OPTIMIZE capability.
#   ✅ FIX: Kept parquet for landing zone (matches existing pipeline),
#   but added optimal write settings (snappy, target file size).
#
# COMPUTE RECOMMENDATION (see bottom of file):
#   Current: 16GB 4-core single node
#   Recommended: 32GB 8-core (Standard_DS4_v2 / r5.2xlarge)
#   Cost delta: ~$0.10/hr more — pays back immediately in time saved.
#   For 1B records: use Spot instances (historical = idempotent = spot-safe).
# =============================================================================

import sys
import os
import pandas as pd
from datetime import datetime

In [0]:
# CELL 1 — Widgets
dbutils.widgets.text("customer_id",      "")
dbutils.widgets.text("object_name",      "")
dbutils.widgets.text("source_system",    "")
dbutils.widgets.text("bq_native_table",  "v4c-bigquery.v4c_bigquery_dataset.event_raw")

customer_id    = dbutils.widgets.get("customer_id")
object_name    = dbutils.widgets.get("object_name")
source_system  = dbutils.widgets.get("source_system")
bq_native_table = dbutils.widgets.get("bq_native_table")
load_type      = "historical"

In [0]:
# CELL 2 — Config: single source of truth (replaces DataLakeConfig import)
%run ../../config/pipeline_config

In [0]:
# Load audit logger for error-only logging
%run ../../utils/audit_logger

In [0]:
# CELL 3 — Performance-tuned Spark config

# Ensure audit tables exist (error-only logging uses these)
initialize_audit_tables()

# ✅ FIX 2: shuffle partitions = 8x cores (32 for 4-core, 64 for 8-core)
num_cores = int(spark.conf.get("spark.default.parallelism", "8"))
shuffle_partitions = max(32, num_cores * 8)

spark.conf.set("spark.sql.shuffle.partitions",                         str(shuffle_partitions))

# ✅ FIX 6: Enable AQE — auto-coalesces small partitions, handles skew
spark.conf.set("spark.sql.adaptive.enabled",                           "true")
spark.conf.set("spark.sql.adaptive.coalescePartitions.enabled",        "true")
spark.conf.set("spark.sql.adaptive.coalescePartitions.minPartitionNum","1")
spark.conf.set("spark.sql.adaptive.skewJoin.enabled",                  "true")

# ✅ Target file size: 128MB per output parquet file (optimal for S3 + downstream reads)
spark.conf.set("spark.sql.files.maxRecordsPerFile",                    "0")  # let AQE decide
spark.conf.set("spark.databricks.delta.optimizeWrite.enabled",         "true")  # if writing delta later

# BigQuery connector parallelism settings
# ✅ FIX 5: Tell BQ connector to create more parallel read streams
spark.conf.set("spark.datasource.bigquery.parallelism",                str(num_cores * 4))

# S3 write performance
spark.conf.set("spark.hadoop.fs.s3a.connection.maximum",               "200")
spark.conf.set("spark.hadoop.fs.s3a.fast.upload",                      "true")
spark.conf.set("spark.hadoop.fs.s3a.multipart.size",                   "134217728")  # 128MB parts

print(f"Cores detected          : {num_cores}")
print(f"Shuffle partitions set  : {shuffle_partitions}")
print(f"AQE enabled             : true")

In [0]:
# CELL 4 — Path + credentials
# Use landing_path() from pipeline_config (loaded via %run)
now = datetime.now()
historical_s3_path = f"{landing_path(source_system, customer_id, object_name, load_type)}{now.strftime('%Y/%m/%d/%H%M%S')}/"

try:
    gcp_secret = dbutils.secrets.get(scope="gcp_auth", key="bq_key")
except Exception as e:
    raise Exception(f"❌ Failed to retrieve GCP credentials: {e}")

gcp_project_id = bq_native_table.split(".")[0]

print(f"🚀 Starting OPTIMIZED BigQuery Historical Load")
print(f"   Customer   : {customer_id}")
print(f"   Object     : {object_name}")
print(f"   BQ Table   : {bq_native_table}")
print(f"   S3 Target  : {historical_s3_path}")

In [0]:
# CELL 5 — SINGLE parallel read (replaces the sequential chunk loop)
#
# ✅ FIX 1 + 5: One spark.read call. The BQ Storage Read API automatically
# creates parallel streams (one per core × parallelism multiplier).
# No Python loop. No sequential chunk overhead. No multiple connections.
#
# The connector internally uses BigQuery's ReadRows API which:
#   - Splits the table into parallel shards server-side
#   - Streams each shard to a Spark partition simultaneously
#   - Uses Arrow columnar format for fast deserialization
#
# For a 1B record table with 4 cores + parallelism=16:
#   BQ creates 16 read streams → 16 Spark tasks in parallel
#   Each task reads ~62.5M records simultaneously
#   Total time: max(stream_time) instead of sum(chunk_time)

# Track duration even on failure (must be outside try)
t_start = datetime.now()

try:
    print("Reading from BigQuery using parallel Storage Read API...")

    df = (
        spark.read
        .format("bigquery")
        .option("credentials",           gcp_secret)
        .option("parentProject",          gcp_project_id)
        .option("project",                gcp_project_id)
        .option("table",                  bq_native_table)
        # ✅ Parallel streams: num_cores × 4 = 16 for 4-core, 32 for 8-core
        .option("preferredMinParallelism", str(num_cores * 4))
        # ✅ Arrow format is 3-5x faster than Avro for columnar data
        .option("readDataFormat",          "ARROW")
        # ✅ Push filter down to BQ to avoid reading unnecessary data
        # Only set this if you want a specific date range:
        # .option("filter", "event_timestamp >= '2024-01-01T00:00:00Z'")
        .load()
    )

    # ✅ FIX 3: Don't repartition(2). Let AQE + target file size control output.
    # coalesce only if you need to reduce small files — AQE handles this better.
    # Target: ~128MB parquet files. For 1B records @ 100 bytes avg = 100GB
    # 100GB / 128MB = ~800 output files. Set explicitly if AQE is not enough:
    target_partitions = max(shuffle_partitions, int(num_cores * 4))

    (
        df
        .repartition(target_partitions)   # even distribution across cores
        .write
        .mode("append")
        .format("parquet")
        .option("compression", "snappy")
        # ✅ Prevents tiny files: max 1M rows per file as a safety net
        .option("maxRecordsPerFile", "2000000")
        .save(historical_s3_path)
    )

    t_end  = datetime.now()
    elapsed = (t_end - t_start).total_seconds()
    print(f"✅ Write complete in {elapsed:.1f}s  ({elapsed/60:.1f} min)")

    # CELL 6 — Get actual row count + watermark from the written data
    # (cheaper than querying BQ again — read from what we just wrote)
    print("Computing watermark from written data...")

    df_written = spark.read.parquet(historical_s3_path)
    total_rows = df_written.count()
    print(f"✅ Total rows written: {total_rows:,}")

    # Get max timestamp for watermark — read from local parquet, not BQ
    from pyspark.sql.functions import max as spark_max
    max_ts_row = df_written.select(spark_max("event_timestamp").alias("max_ts")).collect()[0]
    end_date   = pd.to_datetime(max_ts_row["max_ts"]).replace(tzinfo=None)

    # CELL 7 — Exact Watermark (no lookback — incremental uses strict >)
    # Store the exact max_ts. The incremental query uses WHERE ts > watermark,
    # so the last record is excluded and the first new record is included.
    # Zero duplicates, zero gaps.
    print(f"Storing exact watermark: {end_date}")

    spark.sql("""
        CREATE TABLE IF NOT EXISTS ingestion_metadata.watermarks (
            source_system              STRING,
            object_name                STRING,
            last_processed_timestamp   TIMESTAMP
        ) USING DELTA
    """)

    spark.sql(f"""
        MERGE INTO ingestion_metadata.watermarks AS target
        USING (
            SELECT '{source_system}'  AS source_system,
                   '{object_name}'    AS object_name,
                   TIMESTAMP('{end_date}') AS last_processed_timestamp
        ) AS source
        ON  target.source_system = source.source_system
        AND target.object_name   = source.object_name
        WHEN MATCHED THEN
            UPDATE SET target.last_processed_timestamp = source.last_processed_timestamp
        WHEN NOT MATCHED THEN
            INSERT (source_system, object_name, last_processed_timestamp)
            VALUES (source.source_system, source.object_name, source.last_processed_timestamp)
    """)

    print(f"✅ Watermark set to exact max_ts: {end_date}")
    print(f"\n{'='*55}")
    print(f"  Historical load COMPLETE")
    print(f"  Records  : {total_rows:,}")
    print(f"  Duration : {elapsed:.1f}s ({elapsed/60:.1f} min)")
    print(f"  Rate     : {total_rows/elapsed:,.0f} records/sec")
    print(f"{'='*55}")

except Exception as e:
    elapsed_ms = int((datetime.now() - t_start).total_seconds() * 1000)
    print(f"❌ Pipeline Failed: {e}")
    # ✅ Error-only audit logging — log to bronze.logs.audit_logs
    log_audit(
        job_name       = "BQ_Historical_Load",
        customer_id    = customer_id,
        status         = "failure",
        layer          = "bronze",
        alert_type     = "FAILURE",
        error_type     = type(e).__name__,
        error_reason   = str(e)[:500],
        duration_ms    = elapsed_ms,
    )
    raise e

# =============================================================================
# COMPUTE RECOMMENDATION
# =============================================================================
# Current setup: 16GB 4-core, 1 node — 1hr for 120M records
#
# ┌─────────────────────────────────────────────────────────────┐
# │ RECOMMENDED for 1B records                                      │
# │                                                                 │
# │ Option A — Single large node (simplest, no cluster overhead)    │
# │   AWS:   r5.4xlarge  (16 cores, 128GB RAM)   ~$1.00/hr on-demand│
# │   Azure: Standard_E16s_v3 (16 cores, 128GB)  ~$1.00/hr         │
# │   DBR:   16.0 LTS, single node mode                             │
# │   Expected: 1B records in ~20-30 min                            │
# │                                                                 │
# │ Option B — Cost-optimized (Spot + small cluster)                │
# │   1 driver + 2 workers, each r5.2xlarge (8c/64GB)              │
# │   Spot pricing: ~$0.30/hr total                                  │
# │   Historical loads are idempotent → 100% safe on Spot           │
# │   Expected: 1B records in ~15-20 min                            │
# │                                                                 │
# │ Option C — Minimal upgrade from current (cheapest path)         │
# │   r5.xlarge → r5.2xlarge (8c/64GB)  ~$0.20/hr more             │
# │   Expected: 1B records in ~35-45 min                            │
# │                                                                 │
# │ DO NOT use compute with < 8 cores for 1B BQ records.            │
# │ The BQ Storage Read API parallelism is capped by core count.    │
# └─────────────────────────────────────────────────────────────┘
# =============================================================================

In [0]:
# import sys
# import os
# import pandas as pd 
# from datetime import datetime, timedelta
# from pyspark.sql.functions import min as spark_min, max as spark_max

# # 1. Widget Setup
# dbutils.widgets.text("customer_id", "")
# dbutils.widgets.text("object_name", "")
# dbutils.widgets.text("source_system", "")
# dbutils.widgets.text("bq_native_table", "v4c-bigquery.v4c_bigquery_dataset.event_raw")

# customer_id = dbutils.widgets.get("customer_id")
# object_name = dbutils.widgets.get("object_name")
# source_system = dbutils.widgets.get("source_system")
# bq_native_table = dbutils.widgets.get("bq_native_table")
# load_type = "historical" # Defined for config context

# # --------------------------------------------------
# # 2. Dynamic Config Import
# # --------------------------------------------------
# # Pointing to the root where 'utils' folder resides
# project_root = "/Workspace/Users/ayush.gunjal@hginsights.com/HGI-Lakehouse-Pipeline"
# abs_project_root = os.path.abspath(project_root)

# if abs_project_root not in sys.path:
#     sys.path.append(abs_project_root)

# from utils.config import DataLakeConfig

# # Instantiate Config
# config = DataLakeConfig(source_system, customer_id, object_name, load_type)

# # Optimize shuffle for your 2-core single node
# spark.conf.set("spark.sql.shuffle.partitions", 2)

# # Define Historical Path using Config Helper
# now = datetime.now()
# historical_s3_path = config.get_historical_landing_path(source_system, customer_id, object_name, now)

# print(f"🚀 Starting FAST NATIVE BigQuery Historical Load for {customer_id} - {object_name}")
# print(f"Target Path: {historical_s3_path}")

# gcp_project_id = bq_native_table.split(".")[0]

# # 3. Securely Retrieve GCP Credentials
# try:
#     gcp_secret = dbutils.secrets.get(scope="gcp_auth", key="bq_key")
# except Exception as e:
#     raise Exception("❌ Failed to retrieve GCP credentials. Ensure the secret scope 'gcp_auth' and key 'bq_key' exist.")

# try:
#     # 4. Find Min/Max (Direct gRPC Read)
#     print("Fetching timeline boundaries from BigQuery...")
#     bounds_df = spark.read \
#         .format("bigquery") \
#         .option("credentials", gcp_secret) \
#         .option("parentProject", gcp_project_id) \
#         .option("project", gcp_project_id) \
#         .option("table", bq_native_table) \
#         .load() \
#         .select(
#             spark_min("event_timestamp").alias("min_ts"),
#             spark_max("event_timestamp").alias("max_ts")
#         )
        
#     bounds = bounds_df.collect()[0]
#     start_date_raw = bounds["min_ts"]
#     end_date_raw = bounds["max_ts"]
    
#     if start_date_raw is None or end_date_raw is None:
#         print("No data found in source.")
#         dbutils.notebook.exit("0") 
    
#     start_date = pd.to_datetime(start_date_raw).replace(tzinfo=None)
#     end_date = pd.to_datetime(end_date_raw).replace(tzinfo=None)
        
#     print(f"Data range: {start_date} to {end_date}")

#     # 5. Divide timeline into chunks
#     intervals = []
#     current_start = start_date
#     chunk_interval = timedelta(hours=12) 
    
#     while current_start <= (end_date + chunk_interval):
#         current_end = current_start + chunk_interval
#         str_start = current_start.strftime("%Y-%m-%dT%H:%M:%S.%f")[:-3] + "Z"
#         str_end = current_end.strftime("%Y-%m-%dT%H:%M:%S.%f")[:-3] + "Z"
#         intervals.append((str_start, str_end))
#         current_start = current_end

#     print(f"Divided timeline into {len(intervals)} 12-hour chunks.")

#     # 6. Process Sequentially using Native Connector
#     for start_str, end_str in intervals:
#         try:
#             print(f"Processing chunk: {start_str} to {end_str}...")
#             bq_filter_logic = f"event_timestamp >= '{start_str}' AND event_timestamp < '{end_str}'"
            
#             df_chunk = spark.read \
#                 .format("bigquery") \
#                 .option("credentials", gcp_secret) \
#                 .option("parentProject", gcp_project_id) \
#                 .option("project", gcp_project_id) \
#                 .option("filter", bq_filter_logic) \
#                 .option("table", bq_native_table) \
#                 .load()
            
#             df_chunk.repartition(2).write.mode("append") \
#                 .format("parquet") \
#                 .option("compression","snappy") \
#                 .save(historical_s3_path)
            
#             print(f"✅ Successfully written chunk: {start_str}")
            
#         except Exception as e:
#             print(f"❌ Failed chunk {start_str}: {str(e)}")
#             raise e

#     print("✅ Historical Data Load FINISHED!")

#     # 7. Initialize Watermark
#     print(f"Initializing watermark to {end_date} minus 1 minute...")
    
#     spark.sql("""
#         CREATE TABLE IF NOT EXISTS ingestion_metadata.watermarks (
#             source_system STRING,
#             object_name STRING,
#             last_processed_timestamp TIMESTAMP
#         ) USING DELTA
#     """)
    
#     spark.sql(f"""
#         CREATE OR REPLACE TEMP VIEW new_watermarks AS
#         SELECT source_system, object_name, last_processed_timestamp
#         FROM ingestion_metadata.watermarks
#         WHERE NOT (source_system = '{source_system}' AND object_name = '{object_name}')
#         UNION ALL
#         SELECT '{source_system}' AS source_system, '{object_name}' AS object_name, TIMESTAMP('{end_date}') - INTERVAL 1 MINUTE AS last_processed_timestamp
#     """)
    
#     spark.sql("INSERT OVERWRITE TABLE ingestion_metadata.watermarks SELECT * FROM new_watermarks")
#     print("✅ Watermark initialized successfully.")

# except Exception as e:
#     print(f"❌ Pipeline Failed: {str(e)}")
#     raise e